# Planck CMB Interactive Explorer

Interactive all-sky visualization of the **Planck 2018 (PR3) component-separated Cosmic Microwave Background**.

**Default map:** SMICA-NOSZ temperature map — Milky Way foregrounds component-separated, with the thermal Sunyaev–Zel'dovich signal additionally nulled.

Use the controls to change the component-separation pipeline, coordinate system, projection, color scale, temperature range, smoothing, and display resolution.

> This is a map of tiny CMB temperature anisotropies, not a photograph of matter in the modern universe. The mean 2.725 K background and the motion dipole are conventionally removed. Residual foreground contamination remains strongest near the Galactic plane.


In [ ]:
!pip -q install healpy ipywidgets astropy requests

from google.colab import output
output.enable_custom_widget_manager()

print("Installation complete. Run the next cell.")


In [ ]:
import gc
import requests
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp

from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as widgets

CACHE_DIR = Path("/content/planck_cmb_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = (
    "https://irsa.ipac.caltech.edu/data/Planck/release_3/"
    "all-sky-maps/maps/component-maps/cmb/"
)

MAPS = {
    "SMICA-NOSZ — recommended (384 MB)": {
        "file": "COM_CMB_IQU-smica-nosz_2048_R3.00_full.fits",
        "field": 0,
        "note": "SMICA temperature map with thermal SZ nulled; compact intensity-only file."
    },
    "SMICA (1.9 GB)": {
        "file": "COM_CMB_IQU-smica_2048_R3.00_full.fits",
        "field": 0,
        "note": "Standard Planck SMICA temperature solution."
    },
    "NILC (1.5 GB)": {
        "file": "COM_CMB_IQU-nilc_2048_R3.00_full.fits",
        "field": 0,
        "note": "Needlet Internal Linear Combination solution."
    },
    "Commander (1.5 GB)": {
        "file": "COM_CMB_IQU-commander_2048_R3.00_full.fits",
        "field": 0,
        "note": "Bayesian parametric component-separation solution."
    },
    "SEVEM (1.9 GB)": {
        "file": "COM_CMB_IQU-sevem_2048_R3.00_full.fits",
        "field": 0,
        "note": "Template-cleaned component-separation solution."
    },
}

_loaded = {"key": None, "map_uK": None}

def download_with_progress(url, destination):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 1_000_000:
        return destination

    tmp = destination.with_suffix(destination.suffix + ".part")
    existing = tmp.stat().st_size if tmp.exists() else 0
    headers = {"Range": f"bytes={existing}-"} if existing else {}

    with requests.get(url, stream=True, timeout=120, headers=headers) as response:
        response.raise_for_status()
        total_new = int(response.headers.get("content-length", 0))
        total = existing + total_new
        mode = "ab" if existing and response.status_code == 206 else "wb"
        if mode == "wb":
            existing = 0

        downloaded = existing
        last_pct = -1
        with open(tmp, mode) as f:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if not chunk:
                    continue
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    pct = int(100 * downloaded / total)
                    if pct >= last_pct + 5:
                        print(f"Download: {pct:3d}% ({downloaded/1e6:,.0f} MB / {total/1e6:,.0f} MB)")
                        last_pct = pct

    tmp.replace(destination)
    return destination

def get_map(map_key):
    if _loaded["key"] == map_key and _loaded["map_uK"] is not None:
        return _loaded["map_uK"]

    info = MAPS[map_key]
    path = CACHE_DIR / info["file"]
    url = BASE_URL + info["file"]

    print(f"Selected: {map_key}")
    print(info["note"])
    print(f"Cache: {path}")
    download_with_progress(url, path)

    print("Reading HEALPix temperature field...")
    cmb_K = hp.read_map(path, field=info["field"], dtype=np.float32)
    cmb_uK = cmb_K.astype(np.float32) * 1e6
    cmb_uK[~np.isfinite(cmb_uK)] = hp.UNSEEN

    _loaded["key"] = map_key
    _loaded["map_uK"] = cmb_uK
    gc.collect()
    return cmb_uK

def prepare_map(cmb_uK, nside_display, smoothing_deg):
    source_nside = hp.get_nside(cmb_uK)
    target_nside = min(int(nside_display), source_nside)

    work = cmb_uK
    if target_nside != source_nside:
        work = hp.ud_grade(work, nside_out=target_nside, power=0).astype(np.float32)

    if smoothing_deg > 0:
        work = hp.smoothing(work, fwhm=np.radians(float(smoothing_deg))).astype(np.float32)

    return work

def render_map(map_key, coordinates, projection, cmap, amplitude_uK,
               smoothing_deg, nside_display, scale_mode,
               show_graticule, show_galactic_plane):
    clear_output(wait=True)

    try:
        cmb_uK = get_map(map_key)
        work = prepare_map(cmb_uK, nside_display, smoothing_deg)

        coord_lookup = {
            "Galactic": "G",
            "Celestial / Equatorial": "C",
            "Ecliptic": "E",
        }
        coord = coord_lookup[coordinates]
        valid = work[(work != hp.UNSEEN) & np.isfinite(work)]
        rms = float(np.std(valid))
        mean = float(np.mean(valid))
        p01, p99 = np.percentile(valid, [1, 99])
        norm = None if scale_mode == "Linear" else "hist"

        title = (
            "PLANCK 2018 CMB TEMPERATURE ANISOTROPY\n"
            f"{map_key.split(' —')[0].split(' (')[0]} | {coordinates} | "
            f"smoothing {smoothing_deg:.1f}°"
        )

        plt.figure(figsize=(16, 8.7), dpi=120)
        common = dict(
            map=work, coord=coord, title=title, unit="µK_CMB", cmap=cmap,
            min=-float(amplitude_uK), max=float(amplitude_uK), norm=norm,
            badcolor="0.25", bgcolor="0.03", cbar=True, notext=False, hold=True
        )

        if projection == "Mollweide":
            hp.mollview(**common)
        else:
            hp.cartview(**common, lonra=[-180, 180], latra=[-90, 90])

        if show_graticule:
            hp.graticule(dpar=30, dmer=30, color="white", alpha=0.25)

        if show_galactic_plane:
            lon = np.linspace(-180, 180, 721)
            lat = np.zeros_like(lon)
            hp.projplot(lon, lat, lonlat=True, coord=["G", coord],
                        linestyle="--", linewidth=1.2, color="white", alpha=0.75)

        fig = plt.gcf()
        fig.text(0.5, 0.025,
                 "Temperature fluctuations around the mean CMB temperature; monopole and Solar-system motion dipole removed. "
                 "Milky Way foregrounds component-separated, not perfectly erased.",
                 ha="center", va="bottom", fontsize=10)
        fig.text(0.01, 0.025,
                 f"Displayed Nside={hp.get_nside(work)} | mean={mean:.2f} µK | RMS={rms:.2f} µK | "
                 f"1–99%={p01:.1f}…{p99:.1f} µK",
                 ha="left", va="bottom", fontsize=9)
        plt.show()

    except Exception as exc:
        print("The map could not be displayed.")
        print(type(exc).__name__ + ":", exc)
        print("Try the default SMICA-NOSZ map first. Large alternatives require substantial RAM and disk space.")

style = {"description_width": "190px"}
layout = widgets.Layout(width="520px")

map_dropdown = widgets.Dropdown(options=list(MAPS.keys()), value="SMICA-NOSZ — recommended (384 MB)", description="Planck solution:", style=style, layout=layout)
coord_dropdown = widgets.Dropdown(options=["Galactic", "Celestial / Equatorial", "Ecliptic"], value="Galactic", description="Coordinate system:", style=style, layout=layout)
projection_dropdown = widgets.Dropdown(options=["Mollweide", "Cartesian"], value="Mollweide", description="Projection:", style=style, layout=layout)
cmap_dropdown = widgets.Dropdown(options=["coolwarm", "RdBu_r", "turbo", "inferno", "viridis", "plasma", "cividis"], value="coolwarm", description="Color map:", style=style, layout=layout)
amplitude_slider = widgets.IntSlider(value=300, min=50, max=600, step=10, description="Color range ±µK:", continuous_update=False, style=style, layout=layout)
smoothing_slider = widgets.FloatSlider(value=0.0, min=0.0, max=5.0, step=0.25, description="Gaussian smoothing:", readout_format=".2f", continuous_update=False, style=style, layout=layout)
nside_dropdown = widgets.Dropdown(options=[("Fast — Nside 256", 256), ("Detailed — Nside 512", 512), ("High — Nside 1024", 1024)], value=512, description="Display resolution:", style=style, layout=layout)
scale_dropdown = widgets.Dropdown(options=["Linear", "Histogram equalized"], value="Linear", description="Contrast scaling:", style=style, layout=layout)
graticule_checkbox = widgets.Checkbox(value=True, description="Show coordinate grid", indent=False)
plane_checkbox = widgets.Checkbox(value=False, description="Mark Galactic plane", indent=False)

controls = widgets.VBox([
    widgets.HTML("<h2 style='margin:0'>🌌 Planck CMB Interactive Explorer</h2><p style='margin-top:4px'>Foreground-cleaned full-sky temperature anisotropy.</p>"),
    map_dropdown, coord_dropdown, projection_dropdown, cmap_dropdown,
    amplitude_slider, smoothing_slider, nside_dropdown, scale_dropdown,
    widgets.HBox([graticule_checkbox, plane_checkbox]),
])

output = widgets.interactive_output(render_map, {
    "map_key": map_dropdown,
    "coordinates": coord_dropdown,
    "projection": projection_dropdown,
    "cmap": cmap_dropdown,
    "amplitude_uK": amplitude_slider,
    "smoothing_deg": smoothing_slider,
    "nside_display": nside_dropdown,
    "scale_mode": scale_dropdown,
    "show_graticule": graticule_checkbox,
    "show_galactic_plane": plane_checkbox,
})

display(controls, output)


## How to interpret the map

- **Red and blue do not mean ordinary hot and cold matter today.** They show temperature deviations of only a few hundred microkelvin around the 2.725 K mean CMB temperature.
- The structures are a projected view of primordial fluctuations at the surface of last scattering, roughly 380,000 years after the Big Bang.
- The Solar System's motion dipole is removed from standard science maps.
- “Milky Way subtracted” means foreground component separation—not perfect deletion. Residuals and processing uncertainty remain, especially near the Galactic plane.
- **SMICA-NOSZ** is the recommended first view here because it is compact and suppresses the thermal SZ contribution while retaining the primary CMB temperature field.

Official data: NASA/IPAC Infrared Science Archive, Planck Public Data Release 3.
